# Seminario AIoT · Machine Learning con temperaturas de una laguna
**UPV · DISCA · 30–40 minutos**  
Tres mini-ejercicios: **regresión, clasificación y clustering**. Utilizaremos una misma serie temporal y código breve, con algunas decisiones para completar por el alumnado.

**Datos:** repositorio [SII-UPV](https://github.com/chemacecilia/SII-UPV), carpeta `datos/`. El notebook intenta detectar automáticamente archivos CSV o Excel y columnas de fecha/temperatura. **Comprueba la selección antes de continuar.** Si no encuentra el repositorio, puedes cargar tu archivo manualmente.

> Importante: no se ha podido verificar el esquema de los ficheros individuales. Por ello, la carga es adaptable; no se inventan columnas ni mediciones reales.

**Fichero confirmado:** `datos/laguna_datos_horarios_rellenos.csv`. Columnas: `fecha`, `temperatura`, `ambiente`, `nivel`. La fecha tiene formato `dd/mm/aaaa H:MM`.


## 0. Preparación (5 min)
**Requisitos:** Python 3.10+, pandas, numpy, matplotlib, scikit-learn y, para Excel, openpyxl. En Jupyter ejecuta la siguiente celda si falta alguna dependencia.

In [ ]:
# Descomenta si es necesario:
# %pip install pandas numpy matplotlib scikit-learn openpyxl

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, accuracy_score, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.figsize"] = (9, 4)
print("Entorno preparado")

### 0.1 Datos del repositorio
Clona el repositorio `SII-UPV` y coloca este notebook en su raíz o en `codigo/`. El fichero utilizado es **`datos/laguna_datos_horarios_rellenos.csv`**. La carga admite que el separador sea tabulador, coma o punto y coma.

In [ ]:
RUTAS = [Path("datos/laguna_datos_horarios_rellenos.csv"),
         Path("../datos/laguna_datos_horarios_rellenos.csv"),
         Path("SII-UPV/datos/laguna_datos_horarios_rellenos.csv")]
ruta = next((p for p in RUTAS if p.is_file()), None)
if ruta is None:
    raise FileNotFoundError("No encuentro datos/laguna_datos_horarios_rellenos.csv. Ejecuta el notebook desde la raíz del repositorio o desde codigo/.")
print("Fichero encontrado:", ruta)

### 0.2 Leer las cuatro columnas reales
El fichero contiene `fecha`, `temperatura`, `ambiente` y `nivel`. En la muestra facilitada los campos están separados por tabulaciones; `sep=None` permite también otros separadores habituales.

In [ ]:
df = pd.read_csv(ruta, sep=None, engine="python", encoding="utf-8-sig")
df.columns = df.columns.str.strip().str.lower()
columnas = ["fecha", "temperatura", "ambiente", "nivel"]
faltan = set(columnas) - set(df.columns)
if faltan:
    raise ValueError(f"Faltan columnas: {sorted(faltan)}. Columnas leídas: {list(df.columns)}")
print("Observaciones originales:", len(df))
display(df[columnas].head())

In [ ]:
### 0.3 Preparar la serie temporal
Convertimos la fecha con `dayfirst=True`, ordenamos por hora y generamos la temperatura de la **hora siguiente**. Para no confundir saltos temporales con predicciones a una hora, eliminamos pares cuya separación no sea de una hora.

In [ ]:
serie = df[["fecha", "temperatura", "ambiente", "nivel"]].copy()
serie["fecha"] = pd.to_datetime(serie["fecha"], dayfirst=True, errors="coerce")
for col in ["temperatura", "ambiente", "nivel"]:
    serie[col] = pd.to_numeric(serie[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")
serie = (serie.dropna(subset=["fecha", "temperatura", "ambiente", "nivel"])
         .sort_values("fecha").drop_duplicates("fecha").reset_index(drop=True))
serie["temp_anterior"] = serie["temperatura"].shift(1)
serie["temp_siguiente"] = serie["temperatura"].shift(-1)
serie["variacion"] = serie["temperatura"] - serie["temp_anterior"]
# Mantener solo registros con hora anterior y posterior realmente consecutivas.
valida = (serie["fecha"].diff() == pd.Timedelta(hours=1)) & (serie["fecha"].shift(-1) - serie["fecha"] == pd.Timedelta(hours=1))
serie = serie.loc[valida].dropna().reset_index(drop=True)
if len(serie) < 30:
    raise ValueError("Hay menos de 30 filas consecutivas completas; revisa los datos")
print("Filas utilizables:", len(serie))
display(serie.head())
plt.figure(figsize=(10, 3))
plt.plot(serie["fecha"], serie["temperatura"], lw=0.8)
plt.title("Temperatura de la laguna")
plt.xlabel("Fecha"); plt.ylabel("Temperatura (°C)"); plt.tight_layout(); plt.show()

## 1. Regresión · Temperatura de la próxima hora (10 min)
**Pregunta:** con la temperatura del agua, la temperatura ambiente y el nivel actuales, ¿podemos estimar la temperatura del agua una hora después?

**Modelo:** regresión lineal. Separa cronológicamente 80 % para entrenar y 20 % para probar; compara con la predicción trivial «mañana = hoy» (aquí, «hora siguiente = hora actual»).

In [ ]:
variables = ["temperatura", "ambiente", "nivel"]
X = serie[variables]
y = serie["temp_siguiente"]
corte = int(len(serie) * 0.8)
X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]
modelo_reg = LinearRegression().fit(X_train, y_train)
pred_reg = modelo_reg.predict(X_test)
print(f"MAE regresión: {mean_absolute_error(y_test, pred_reg):.3f} °C")
print(f"MAE persistencia: {mean_absolute_error(y_test, X_test['temperatura']):.3f} °C")
plt.plot(y_test.to_numpy()[:80], label="Real")
plt.plot(pred_reg[:80], label="Regresión")
plt.xlabel("Horas de prueba"); plt.ylabel("Temperatura (°C)")
plt.title("Regresión: temperatura real y predicha")
plt.legend(); plt.show()

**Para pensar:** ¿supera el modelo lineal a la predicción ingenua de *persistencia*? ¿Qué ocurre si la frecuencia de muestreo es muy alta?

**Mini-reto (opcional):** cambia el modelo para predecir **tres mediciones después** con `shift(-3)` y compara el MAE.

## 2. Clasificación · ¿Subirá la temperatura? (10 min)
**Pregunta:** ¿subirá la temperatura del agua en la siguiente hora?

La etiqueta es **1 = sube** y **0 = no sube**. Como entradas utilizamos `temperatura`, `ambiente`, `nivel` y la variación de temperatura durante la hora anterior. La división entrenamiento/prueba sigue siendo cronológica.

In [ ]:
serie["sube"] = (serie["temp_siguiente"] > serie["temperatura"]).astype(int)
X = serie[["temperatura", "ambiente", "nivel", "variacion"]]
y = serie["sube"]
X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]
print("Distribución en entrenamiento:", y_train.value_counts().to_dict())
if y_train.nunique() < 2:
    print("El conjunto de entrenamiento solo tiene una clase: cambia el intervalo o la definición del objetivo.")
else:
    modelo_clf = LogisticRegression(max_iter=1000)
    modelo_clf.fit(X_train, y_train)
    pred_clf = modelo_clf.predict(X_test)
    print(f"Exactitud: {accuracy_score(y_test, pred_clf):.2%}")
    print(f"Referencia clase mayoritaria: {max(y_test.mean(), 1-y_test.mean()):.2%}")
    ConfusionMatrixDisplay.from_predictions(y_test, pred_clf, display_labels=["No sube", "Sube"], cmap="Blues")
    plt.title("Clasificación: matriz de confusión")
    plt.show()

**Para pensar:** ¿es buena una exactitud del 80 % si el 90 % de las observaciones no suben? ¿Qué información aporta la matriz de confusión?

**Mini-reto (opcional):** modifica el umbral para que «sube» signifique un aumento superior a **0,1 unidades** (solo si tiene sentido para el sensor).

## 3. Clustering · Estados de la laguna (10 min)
**Pregunta:** ¿se distinguen grupos de observaciones según temperatura del agua y temperatura ambiente?

Aplicaremos **K-means con tres grupos**, sin etiquetas. Estandarizamos las dos variables para que tengan escalas comparables. Los nombres de los grupos son arbitrarios: no equivalen por sí mismos a niveles de riesgo.

In [ ]:
X = serie[["temperatura", "ambiente"]].copy()
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)
modelo_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
serie["grupo"] = modelo_kmeans.fit_predict(X_escalado)
plt.scatter(serie["ambiente"], serie["temperatura"], c=serie["grupo"], cmap="viridis", s=12, alpha=0.65)
plt.xlabel("Temperatura ambiente (°C)"); plt.ylabel("Temperatura del agua (°C)")
plt.title("Clustering: grupos térmicos sin etiquetas")
plt.colorbar(label="Grupo asignado"); plt.show()
print("Número de observaciones por grupo:")
print(serie["grupo"].value_counts().sort_index())
print("Promedios de cada grupo:")
display(serie.groupby("grupo")[["temperatura", "ambiente", "nivel"]].mean().round(2))

**Para pensar:** ¿qué podría representar cada grupo? ¿Se corresponden necesariamente con «normal», «alerta» y «anomalía»? **No:** esa interpretación necesita evidencia externa.

**Mini-reto (opcional):** prueba `n_clusters=2` y `n_clusters=4`. ¿Qué cambia?

## Puesta en común · 5 minutos

1. **Regresión:** ¿qué valor predice el modelo y cómo medimos su error?
2. **Clasificación:** ¿qué etiqueta predice y cómo interpretamos los errores?
3. **Clustering:** ¿qué grupos encuentra y por qué no conocemos su significado de antemano?

**Conclusión:** el tipo de aprendizaje depende de la **pregunta** y de la **información disponible**, no de que el sensor o el conjunto de datos cambien.

**Nota metodológica:** estos son ejemplos didácticos. La frecuencia de muestreo, la estacionalidad, los datos ausentes, la autocorrelación y los cambios de régimen afectan a la evaluación real de modelos temporales.